# 01 — Data Profiling

**Projeto:** iFood Customer Analytics (simulação)  
**Squad:** Customer Intelligence  
**Demanda:** MKT-001  
**Área solicitante:** Marketing & CRM

## Objetivo
Investigar a estrutura, qualidade e consistência dos dados de clientes antes do tratamento.

**Fonte:** `data/raw/mkt_data.csv`  
**Entregável:** `reports/data_quality_report.csv`

**Escopo:** diagnóstico exclusivamente. As transformações definitivas serão realizadas em `02_data_cleaning.ipynb`. Execute este notebook a partir da pasta `notebooks/`.

## 1. Configuração e carregamento

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

RAW_PATH = Path('../data/raw/mkt_data.csv')
REPORTS_PATH = Path('../reports')
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

In [2]:
# Carregamento da fonte original.
df = pd.read_csv(RAW_PATH)
df.head()

,Unnamed: 0,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,education_Graduation,education_Master,education_PhD,MntTotal,MntRegularProds,AcceptedCmpOverall,marital_status,education_level,kids,expenses
0,0,58138.0,0,0,58,635,88,546,172,88,...,3.0,NaN,NaN,1529,1441,0,Single,Graduation,0,1529
1,1,46344.0,1,1,38,11,1,6,2,1,...,3.0,NaN,NaN,21,15,0,Single,Graduation,2,21
2,2,71613.0,0,0,26,426,49,127,111,21,...,3.0,NaN,NaN,734,692,0,Together,Graduation,0,734
3,3,26646.0,1,0,26,11,4,20,10,3,...,3.0,NaN,NaN,48,43,0,Together,Graduation,1,48
4,4,58293.0,1,0,94,173,43,118,46,27,...,NaN,NaN,5.0,407,392,0,Married,PhD,1,407


## 2. Dimensões e tipos de dados

In [3]:
# Verificando as dimensões da base.
df.shape

(2205, 44)

In [4]:
# Entendendo os tipos dos dados
df.info()
df.dtypes.value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2205 entries, 0 to 2204
Data columns (total 44 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            2205 non-null   int64  
 1   Income                2205 non-null   float64
 2   Kidhome               2205 non-null   int64  
 3   Teenhome              2205 non-null   int64  
 4   Recency               2205 non-null   int64  
 5   MntWines              2205 non-null   int64  
 6   MntFruits             2205 non-null   int64  
 7   MntMeatProducts       2205 non-null   int64  
 8   MntFishProducts       2205 non-null   int64  
 9   MntSweetProducts      2205 non-null   int64  
 10  MntGoldProds          2205 non-null   int64  
 11  NumDealsPurchases     2205 non-null   int64  
 12  NumWebPurchases       2205 non-null   int64  
 13  NumCatalogPurchases   2205 non-null   int64  
 14  NumStorePurchases     2205 non-null   int64  
 15  NumWebVisitsMonth    

int64      31
float64    11
object      2
Name: count, dtype: int64

In [5]:
numeric_columns = df.select_dtypes(include='number').columns
numeric_columns

Index(['Unnamed: 0', 'Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines',
       'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response',
       'Age', 'Customer_Days', 'marital_Divorced', 'marital_Married',
       'marital_Single', 'marital_Together', 'marital_Widow',
       'education_2n Cycle', 'education_Basic', 'education_Graduation',
       'education_Master', 'education_PhD', 'MntTotal', 'MntRegularProds',
       'AcceptedCmpOverall', 'kids', 'expenses'],
      dtype='object')

## 3. Investigação de duplicidades
O índice exportado pode mascarar registros idênticos por conteúdo. A remoção definitiva fica para o notebook 02.

In [6]:
# Investigação da qualidade dos dados

# Registros duplicados
df.duplicated().sum()

np.int64(0)

In [7]:
# Verificação do provável índice técnico.
print('Índice único:', df['Unnamed: 0'].is_unique)
print('Duplicatas sem índice técnico:', df.drop(columns=['Unnamed: 0']).duplicated().sum())

Índice único: True
Duplicatas sem índice técnico: 184


## 4. Dados ausentes

In [8]:
# Análise de dados ausentes

nulls = pd.DataFrame({
    'quantity': df.isna().sum(),
    'percentage': df.isna().mean() * 100
})

nulls = nulls[nulls['quantity'] > 0]

nulls.sort_values('percentage', ascending=False)

,quantity,percentage
education_Basic,2151,97.551020
marital_Widow,2129,96.553288
education_2n Cycle,2007,91.020408
marital_Divorced,1975,89.569161
education_Master,1841,83.492063
education_PhD,1729,78.412698
marital_Single,1728,78.367347
marital_Together,1637,74.240363
marital_Married,1351,61.269841
education_Graduation,1092,49.523810


## 5. Variáveis redundantes e derivadas

In [9]:
# Análise de relação entre variaveis 

(df['expenses'] == df['MntTotal']).all()

np.True_

In [10]:
# Análise de relação entre variaveis 

(df['kids'] == df['Kidhome'] + df['Teenhome']).all()

np.True_

## 6. Colunas constantes
Consideramos os valores nulos na contagem de valores distintos.

In [11]:
constant_columns = df.columns[df.nunique(dropna=False) == 1]
constant_columns

Index(['Z_CostContact', 'Z_Revenue'], dtype='object')

In [12]:
# Verificação dos valores distintos nas colunas constantes.
df[constant_columns].drop_duplicates()

,Z_CostContact,Z_Revenue
0,3,11


## 7. Investigação de valores negativos
Valores negativos são sinalizados, não removidos automaticamente.

In [13]:
# Contagem de valores negativos por coluna.

negative_values = (df[numeric_columns] < 0).sum()

negative_values[negative_values > 0]

MntRegularProds    3
dtype: int64

In [14]:
df.loc[df['MntRegularProds'] < 0]

,Unnamed: 0,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,education_Graduation,education_Master,education_PhD,MntTotal,MntRegularProds,AcceptedCmpOverall,marital_status,education_level,kids,expenses
1307,1307,6560.0,0,0,2,67,11,26,4,3,...,NaN,4.0,NaN,111,-151,0,Single,Master,0,111
1780,1780,7144.0,0,2,92,81,4,33,5,2,...,NaN,NaN,5.0,125,-166,0,Single,PhD,2,125
1949,1949,4428.0,0,1,0,16,4,12,2,4,...,3.0,NaN,NaN,38,-283,0,Married,Graduation,1,38


## 8. Validação dos nulos estruturais

In [15]:
# Investigação dos nulos relacionados ao estado civil.

df[
    ['marital_status', 'marital_Single', 'marital_Married']
].head(15)

,marital_status,marital_Single,marital_Married
0,Single,4.0,NaN
1,Single,4.0,NaN
2,Together,NaN,NaN
3,Together,NaN,NaN
4,Married,NaN,5.0
5,Together,NaN,NaN
6,Divorced,NaN,NaN
7,Married,NaN,5.0
8,Together,NaN,NaN
9,Together,NaN,NaN


In [16]:
df[
    ['education_level', 'education_Basic',
     'education_Graduation']
].head(15)

,education_level,education_Basic,education_Graduation
0,Graduation,NaN,3.0
1,Graduation,NaN,3.0
2,Graduation,NaN,3.0
3,Graduation,NaN,3.0
4,PhD,NaN,NaN
5,Master,NaN,NaN
6,Graduation,NaN,3.0
7,PhD,NaN,NaN
8,PhD,NaN,NaN
9,PhD,NaN,NaN


In [17]:
# Verificando se as colunas categóricas principais possuem dados ausentes:

df[['marital_status', 'education_level']].isna().sum()

marital_status     0
education_level    0
dtype: int64

In [18]:
# Validar os nulos estruturais

# Validação das colunas auxiliares de estado civil.

marital_columns = df.filter(regex='^marital_').drop(
    columns=['marital_status']
).columns

for col in marital_columns:
    category = col.replace('marital_', '')

    valid = (
        df[col].notna() == df['marital_status'].eq(category)
    ).all()

    print(f'{col}: {valid}')

marital_Divorced: True
marital_Married: True
marital_Single: True
marital_Together: True
marital_Widow: True


In [19]:
# Validação das colunas auxiliares de escolaridade.

education_columns = df.filter(regex='^education_').drop(
    columns=['education_level']
).columns

for col in education_columns:
    category = col.replace('education_', '')

    valid = (
        df[col].notna() == df['education_level'].eq(category)
    ).all()

    print(f'{col}: {valid}')

education_2n Cycle: True
education_Basic: True
education_Graduation: True
education_Master: True
education_PhD: True


## 9. Relatório de qualidade por coluna
`unique_values` exclui nulos; `is_constant` considera nulos. `negative_count` não se aplica a colunas textuais.

In [20]:
# Criação do relatório de qualidade.

quality_report = pd.DataFrame({
    'data_type': df.dtypes,
    'total_records': len(df),
    'null_count': df.isna().sum(),
    'null_percentage': df.isna().mean() * 100,
    'unique_values': df.nunique(),
    'is_constant': df.nunique(dropna=False) == 1,
    'negative_count': df.select_dtypes(
        include='number'
    ).lt(0).sum()
})

quality_report

,data_type,total_records,null_count,null_percentage,unique_values,is_constant,negative_count
AcceptedCmp1,int64,2205,0,0.000000,2,False,0.0
AcceptedCmp2,int64,2205,0,0.000000,2,False,0.0
AcceptedCmp3,int64,2205,0,0.000000,2,False,0.0
AcceptedCmp4,int64,2205,0,0.000000,2,False,0.0
AcceptedCmp5,int64,2205,0,0.000000,2,False,0.0
AcceptedCmpOverall,int64,2205,0,0.000000,5,False,0.0
Age,int64,2205,0,0.000000,56,False,0.0
Complain,int64,2205,0,0.000000,2,False,0.0
Customer_Days,int64,2205,0,0.000000,662,False,0.0
Income,float64,2205,0,0.000000,1963,False,0.0


In [21]:
# Exportação do relatório de qualidade.
quality_report.to_csv(REPORTS_PATH / 'data_quality_report.csv', index_label='column_name')
print('Relatório exportado:', REPORTS_PATH / 'data_quality_report.csv')

Relatório exportado: ..\reports\data_quality_report.csv


## Data Profiling — Conclusões

A base original possui 2.205 registros e 44 colunas.

### Problemas e características identificados

- 184 registros duplicados por conteúdo após desconsiderar o índice técnico.
- 10 colunas auxiliares com valores nulos estruturais.
- 2 colunas constantes: `Z_CostContact` e `Z_Revenue`.
- 3 registros com valores negativos em `MntRegularProds`.
- Presença de um provável índice técnico: `Unnamed: 0`.
- Redundância entre `expenses` e `MntTotal`.
- A variável `kids` corresponde à soma de `Kidhome` e `Teenhome`.

### Recomendações para Data Cleaning

- Remover o índice técnico.
- Remover registros duplicados por conteúdo, conforme premissa adotada para o desafio.
- Remover as colunas auxiliares de estado civil e escolaridade da base analítica.
- Remover as colunas constantes da base analítica, preservando a fonte original.
- Preservar os registros com valores negativos até validar sua regra de negócio.
- Avaliar a manutenção das variáveis redundantes.

### Entregáveis

- Relatório de qualidade: `reports/data_quality_report.csv`.
- Diagnóstico documentado no notebook.

Nenhuma alteração definitiva foi realizada no arquivo de origem.

**Ressalva:** a ausência de identificador confiável impede confirmar que registros idênticos representam o mesmo cliente. A remoção futura é uma premissa do desafio, e os valores negativos ainda requerem validação da regra de negócio. O padrão de presença dos nulos auxiliares foi confirmado; o significado dos códigos não foi validado com o proprietário dos dados.